In [ ]:
import numpy as np
import pandas as pd
import mab_subjects
import matplotlib.pyplot as plt
from neuropy import plotting

models = mab_subjects.rnn_exps10.all_models
exps = mab_subjects.rnn_exps10.u_on_u + mab_subjects.rnn_exps10.s_on_s

n_figs = len(models) // 5

for f in range(n_figs):
    fig = plotting.Fig(num=f + 1, nrows=5, ncols=5, size=(10, 10))

    fig_models = models[f * 5 : (f + 1) * 5]
    fig_exps = exps[f * 5 : (f + 1) * 5]

    for m, (mdl, exp) in enumerate(zip(fig_models, fig_exps)):

        # Load the data
        task_perf = exp.b2a.get_optimal_choice_probability()
        loss_history = mdl.training_loss_history
        policy_history = mdl.policy_loss_history
        value_history = mdl.value_loss_history
        entropy_history = mdl.entropy_bonus_history

        curve_names = [
            "task_perf",
            "policy_loss",
            "value_loss",
            "entropy_bonus",
            "total_loss",
        ]
        curves = [
            task_perf,
            policy_history,
            value_history,
            entropy_history,
            loss_history,
        ]
        xlabels = ["Trial_id", "Time step", "Time step", "Time step", "Time step"]
        ylabels = [
            "Probability of Optimal Choice",
            "Policy Loss",
            "Value Loss",
            "Entropy bonus",
            "Total Loss",
        ]

        for i in range(len(curves)):

            ax = fig.subplot(fig.gs[m, i])
            ax.plot(curves[i])
            ax.set_xlabel(xlabels[i])
            ax.set_ylabel(ylabels[i])

In [ ]:
import numpy as np

rewards = []
for i in range(200):
    r = 1.0 if np.random.random() < 0.8 else 0.0
    rewards.append(r)

rewards = np.array(rewards)
np.mean(rewards)

In [ ]:
arm1 = np.random.choice([0, 1], size=200, p=[0.2, 0.8])
arm2 = np.random.choice([0, 1], size=200, p=[0.8, 0.2])

### Activation evolution

In [ ]:
import numpy as np
import pandas as pd
import mab_subjects
from banditpy.models import BanditTrainer2Arm
from sklearn.decomposition import PCA
import itertools

exps = (
    mab_subjects.unstruc_rnn.p8020_good_rnn_sess
    + mab_subjects.struc_rnn.p8020_good_rnn_sess
    # + mab_subjects.unstruc_rnn.p100_good_rnn_sess
    # + mab_subjects.struc_rnn.p100_good_rnn_sess
)

probs = np.array([0.2, 0.3, 0.4, 0.6, 0.7, 0.8])
prob_pairs = np.array(list(itertools.permutations(probs, 2)))
prob_diffs = np.diff(prob_pairs, axis=1).flatten()
pca = PCA(n_components=3)

pca_df = []
for e, exp in enumerate(exps):
    print(exp.sub_name)
    model_path = exp.filePrefix.with_suffix(".pt")
    model = BanditTrainer2Arm.load_model(model_path=model_path, device="cpu")
    hidden_states = []
    for pair in prob_pairs:
        hidden_states.append(
            np.array(model.analyze_hidden_states(reward_probs=pair)["hidden_states"])
        )
    hidden_states = np.vstack(hidden_states)
    hidden_pca = pca.fit_transform(hidden_states)
    final_state = hidden_pca[99::100, :]

    pca_dict = dict(
        pca0=final_state[:, 0],
        pca1=final_state[:, 1],
        pca2=final_state[:, 2],
        prob_diff=prob_diffs,
    )
    pca_dict.update(exp.common_kwargs)
    pca_df.append(pd.DataFrame(pca_dict))

pca_df = pd.concat(pca_df, ignore_index=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statplotannot.plots import Fig

fig = Fig(4, 3, fontsize=12)

df = pca_df.copy()
df["difficulty"] = pd.cut(
    pca_df["prob_diff"],
    bins=[-0.7, -0.3, 0, 0.3, 0.7],
    labels=["left", "left-biased", "right-biased", "right"],
)
# df = df[df["difficulty"].isin(["left", "right"])]


for p, paradigm in enumerate(["8020"]):
    df_paradigm = df[df["paradigm"] == paradigm]
    for g, group in enumerate(["unstruc", "struc"]):
        df_group = df_paradigm[df_paradigm["group"] == group]
        ax = fig.subplot(fig.gs[p, g])
        # ax.scatter(
        #     df_group["pca0"],
        #     df_group["pca1"],
        #     df_group["pca2"],
        #     c=df_group["difficulty"].cat.codes,
        #     cmap="coolwarm",
        #     alpha=0.7,
        # )
        sns.scatterplot(
            data=df_group,
            x="pca0",
            y="pca1",
            hue="difficulty",
            palette=["blue", "gray", "gray", "blue"],  # "Set1",
            alpha=0.5,
            size=3,
            ax=ax,
        )

        ax.legend_.remove()

        ax.set_title(f"{paradigm} {group}")

### Mean PCA for all rewards

In [ ]:
import numpy as np
import pandas as pd
import mab_subjects
from banditpy.models import BanditTrainer2Arm
from sklearn.decomposition import PCA
import itertools

exps = (
    mab_subjects.unstruc_rnn.p8020_good_rnn_sess
    + mab_subjects.struc_rnn.p8020_good_rnn_sess
    + mab_subjects.unstruc_rnn.p100_good_rnn_sess
    + mab_subjects.struc_rnn.p100_good_rnn_sess
)

probs = np.array([0.2, 0.3, 0.4, 0.6, 0.7, 0.8])
prob_pairs = np.array(list(itertools.permutations(probs, 2)))
prob_diffs = np.diff(prob_pairs, axis=1).flatten()
pca = PCA(n_components=3)

pca_df = []
for e, exp in enumerate(exps):
    print(exp.sub_name)
    model_path = exp.filePrefix.with_suffix(".pt")
    model = BanditTrainer2Arm.load_model(model_path=model_path, device="cpu")
    hidden_states = []
    for pair in prob_pairs:
        hidden_states.append(
            np.array(model.analyze_hidden_states(reward_probs=pair)["hidden_states"])
        )
    hidden_states = np.vstack(hidden_states)
    hidden_pca = pca.fit_transform(hidden_states)
    final_state = hidden_pca[99::100, :]

    pca_dict = dict(
        pca0=final_state[:, 0],
        pca1=final_state[:, 1],
        pca2=final_state[:, 2],
        prob_diff=prob_diffs,
    )
    pca_dict.update(exp.common_kwargs)
    pca_df.append(pd.DataFrame(pca_dict))

pca_df = pd.concat(pca_df, ignore_index=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statplotannot.plots import Fig

fig = Fig(4, 3, fontsize=12)

df = pca_df.copy()
df["difficulty"] = pd.cut(
    pca_df["prob_diff"],
    bins=[-0.7, -0.3, 0, 0.3, 0.7],
    labels=["left", "left-biased", "right-biased", "right"],
)
# df = df[df["difficulty"].isin(["left", "right"])]


for p, paradigm in enumerate(["8020"]):
    df_paradigm = df[df["paradigm"] == paradigm]
    for g, group in enumerate(["unstruc", "struc"]):
        df_group = df_paradigm[df_paradigm["group"] == group]
        ax = fig.subplot(fig.gs[p, g])
        # ax.scatter(
        #     df_group["pca0"],
        #     df_group["pca1"],
        #     df_group["pca2"],
        #     c=df_group["difficulty"].cat.codes,
        #     cmap="coolwarm",
        #     alpha=0.7,
        # )
        sns.scatterplot(
            data=df_group,
            x="pca0",
            y="pca1",
            hue="difficulty",
            palette=["blue", "gray", "gray", "blue"],  # "Set1",
            alpha=0.5,
            size=3,
            ax=ax,
        )

        ax.legend_.remove()

        ax.set_title(f"{paradigm} {group}")